# Grade-Level Writing Tutor — QLoRA fine-tune on Qwen3-4B (Unsloth)

Runs on **Colab Pro (A100 40GB / L4 24GB)**. Trains Qwen3-4B into the grade 7-8
writing tutor using the dataset built locally (`data/tutor_train_final.jsonl`,
3,667 examples). The 4B base has ~6x the capacity of the local 0.6B, which
targets the two ceilings we hit at 0.6B: multi-turn coherence and grammar
content accuracy. (Config cell has 1.7B / 0.6B fallbacks for a free T4.)

**Flow:** install → upload `tutor_train_final.jsonl` → train QLoRA → sanity chat
→ evaluate on Colab → save adapter (download) → optional push to HF Hub.

**Setup:** Runtime → Change runtime type → **A100 GPU** + tick **Background
execution** before running. Expected wall-clock ~25–40 min (see the last cell).

In [ ]:
# 1. Install Unsloth (pulls compatible torch/trl/peft/bitsandbytes)
%pip install -q "unsloth"
%pip install -q --no-deps "trl<0.15.0" peft accelerate bitsandbytes

In [ ]:
# 2. Upload the dataset. Opens a file picker; upload data/tutor_train_final.jsonl
import os
TRAIN_FILE = 'tutor_train_final.jsonl'
if not os.path.exists(TRAIN_FILE):
    try:
        from google.colab import files
        up = files.upload()
        TRAIN_FILE = list(up.keys())[0]
    except Exception as e:
        raise SystemExit('Upload data/tutor_train_final.jsonl or set TRAIN_FILE. ' + str(e))


In [ ]:
# 3. Config — tutor on Qwen3-4B, maxed for a Colab Pro A100/L4.
MODEL = 'unsloth/Qwen3-4B'           # Pro GPU. Fallbacks: 'unsloth/Qwen3-1.7B', 'unsloth/Qwen3-0.6B'
MAX_SEQ_LEN = 2048                   # long context headroom for multi-turn
LORA_R = 32                          # higher rank = more adapter capacity (was 16 on 0.6B)
LORA_ALPHA = 32
LR = 2e-4
EPOCHS = 3
BATCH = 16                           # A100 40GB handles this at seq 2048; drop to 8 on an L4 (24GB)
GRAD_ACCUM = 1                       # effective batch 16
SEED = 3407
SYSTEM_MINIMAL = 'You are Billy-Bob-Joe, a friendly writing and grammar tutor for a 7th-8th grade student.'
OUTPUT_DIR = 'tutor-4b-v1'
HUB_MODEL_ID = ''  # e.g. 'blackbird0831/qwen3-4b-writing-tutor' to push the merged model

In [ ]:
# 4. Load base model in 4-bit
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

In [ ]:
# 5. Load + tokenize the dataset ourselves (robust across TRL versions on Colab).
#    Each record: {"messages": [{role, content}, ...]}. Prepend a MINIMAL system
#    prompt if absent, apply the chat template, then tokenize to input_ids.
import json
from datasets import Dataset

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

rows = []
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        msgs = rec['messages']
        if not msgs or msgs[0]['role'] != 'system':
            msgs = [{'role': 'system', 'content': SYSTEM_MINIMAL}] + msgs
        rows.append({'messages': msgs})
print('examples:', len(rows))

def tok(ex):
    try:
        text = tokenizer.apply_chat_template(
            ex['messages'], tokenize=False, add_generation_prompt=False, enable_thinking=False)
    except TypeError:
        text = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    out = tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN)
    return {'input_ids': out['input_ids'], 'attention_mask': out['attention_mask']}

ds = Dataset.from_list(rows).map(tok, remove_columns=['messages'])
print('tokenized columns:', ds.column_names)
print('sample length (tokens):', len(ds[0]['input_ids']))

In [ ]:
# 5b. (Recommended) Mount Google Drive so checkpoints survive a disconnect.
#     If you skip this, checkpoints go to local Colab storage (lost on disconnect).
CKPT_DIR = OUTPUT_DIR + '-ckpt'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/' + OUTPUT_DIR + '-ckpt'
    print('Checkpoints -> Google Drive:', CKPT_DIR)
except Exception as e:
    print('Drive not mounted (checkpoints local, lost on disconnect):', e)

In [ ]:
# 6/7. Train (SFT) on the pre-tokenized data. Full-sequence causal LM loss.
#      Periodic checkpoints so a Colab disconnect can resume (re-run to resume).
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling
import os

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

cfg_kwargs = dict(
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=0.05,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=SEED,
    output_dir=CKPT_DIR,
    save_steps=50,
    save_total_limit=2,
    report_to='none',
    max_seq_length=MAX_SEQ_LEN,
    dataset_kwargs={'skip_prepare_dataset': True},  # data is already tokenized
)
try:
    args = SFTConfig(**cfg_kwargs)
except TypeError:
    cfg_kwargs.pop('max_seq_length', None)
    args = SFTConfig(**cfg_kwargs)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    data_collator=collator,
    args=args,
)

resume = os.path.isdir(CKPT_DIR) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT_DIR))
print('Resuming from checkpoint' if resume else 'Starting fresh')
trainer.train(resume_from_checkpoint=resume)

In [ ]:
# 7. Quick sanity chat — the exact things the 0.6B struggled with.
FastLanguageModel.for_inference(model)
import torch
def ask(msgs):
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                        enable_thinking=False, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=inp, max_new_tokens=200, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True).strip()
S = {'role':'system','content':SYSTEM_MINIMAL}
# single-turn probes
for p in ['what is your name?', 'teach me parallel structure', 'what is a gerund?',
          'use bigger words, give me the college version of comma splices']:
    print('YOU:', p); print('TUTOR:', ask([S, {'role':'user','content':p}])); print()
# multi-turn: premature verdict (must NOT confirm a non-answer)
hist = [S, {'role':'user','content':'what is a semicolon'},
        {'role':'assistant','content':'A semicolon joins two complete sentences that are closely related, like: "I was tired; I kept working."'},
        {'role':'user','content':'am I right?'}]
print('MULTITURN am-I-right (no answer given):'); print('TUTOR:', ask(hist))

In [ ]:
# 9. Evaluate on Colab (the 4B won't run on a 4GB laptop, so score it here).
#    Clones the public repo for the eval harness + held-out sets, then runs the
#    deterministic golden gate on THIS adapter.
%pip install -q textstat wordfreq
import os, subprocess, sys
if not os.path.isdir('vocab-locked-writing-tutor'):
    subprocess.run(['git','clone','-q','https://github.com/blackbird-alt/vocab-locked-writing-tutor'], check=True)
REPO = 'vocab-locked-writing-tutor'
BASE = MODEL.replace('unsloth/', 'Qwen/')
ADAPTER = os.path.abspath(OUTPUT_DIR)
# Golden set: greedy, deterministic, judge-free.
subprocess.run([sys.executable, 'eval/golden_check.py',
                '--model', BASE, '--adapter', ADAPTER], cwd=REPO)
print('Golden gate done. For the full base-vs-tuned + grammar battery, run '
      'eval/run_eval.py and scripts/probe_conversations.py from the cloned repo.')

In [ ]:
# 9. Evaluate on Colab (the 4B won't run on a 4GB laptop, so score it here).
#    Clones the public repo for the eval harness + held-out sets, then runs the
#    deterministic golden gate and the 15-question grammar battery on THIS adapter.
%pip install -q textstat wordfreq
import os, subprocess, sys
if not os.path.isdir('vocab-locked-writing-tutor'):
    subprocess.run(['git','clone','-q','https://github.com/blackbird-alt/vocab-locked-writing-tutor'], check=True)
REPO='vocab-locked-writing-tutor'
# Golden set (greedy, deterministic, judge-free):
subprocess.run([sys.executable, f'{REPO}/eval/golden_check.py',
                '--model', MODEL.replace('unsloth/','Qwen/'),
                '--adapter', os.path.abspath(OUTPUT_DIR)], cwd=REPO)
print('
Golden gate done. For the full base-vs-tuned + grammar battery, run
'
      'eval/run_eval.py and scripts/probe_conversations.py from the cloned repo.')

In [ ]:
# 9. (Optional) Merge to 16-bit and push to the Hub for the inference demo + submission.
if HUB_MODEL_ID:
    from huggingface_hub import login
    import os
    login(os.environ.get('HF_TOKEN') or input('HF token: '))
    model.push_to_hub_merged(HUB_MODEL_ID, tokenizer, save_method='merged_16bit')
    print('Pushed', HUB_MODEL_ID)
else:
    print('Set HUB_MODEL_ID to push.')

## Expected runtime on Colab Pro (A100 40GB)

| Step | Time |
|---|---|
| Install Unsloth + deps | 3–5 min |
| Download Qwen3-4B 4-bit (~3 GB) | 3–5 min |
| Tokenize 3,667 examples | <1 min |
| **Train, 3 epochs (cell 6)** | **~10–20 min** (~515 steps, effective batch 16, seq 2048, A100) |
| Colab eval (golden gate) | 3–5 min |
| Save + download / push | 1–3 min |
| **Total** | **~25–40 min** |

On an **L4 (24 GB)** instead of A100: set `BATCH=8` (train ~25–40 min). If cell 6
OOMs, drop `BATCH` again or `MAX_SEQ_LEN` to 1024. The 4B in 4-bit needs ~6–8 GB
for inference, so the demo runs on Colab/a ≥6 GB card, not the 4 GB laptop.